In [ ]:
# Include necessary imports
import pandas as pd
import matplotlib
matplotlib.use("Agg")
from pathlib import Path
from IPython.display import Image, display
from risk_validation.core.metrics.impl import pd as pd_metrics  # noqa: F401 — registers PD metrics
from risk_validation.core.services.pd.metrics_service import PDMetricsService
from risk_validation.core.utils.rag import rag_for_metric

# Load the data
DATA_DIR = Path('..') / 'data'
df = pd.read_csv(DATA_DIR / 'sample.csv')

# Set parameters
params = {
    'y_col': 'default_flag',
    'p_col': 'pred_br',
    'score_col': 'score_pd',
    'period_col': 'QTR',  # Specify period column for trend analysis
    'band_col': 'BAND'  # Include band_col if available
}

# Initialize the metrics service
service = PDMetricsService(['auc_roc', 'gini', 'ks'])

# Compute metrics
develop_year = 2019
validate_year = 2020
results = []
for year in [develop_year, validate_year]:
    df_year = df[df['score_year'] == year]
    result = service.compute(df_year, params)
    results.append(result)
    print(f"Metrics for {year}:\n", result[['metric_id', 'value']])

# Suggested visualizations (display plots for each metric)
from risk_validation.core.utils.plots import plot_gains, plot_roc, plot_ks_cdf_with_maxgap

plot_dir = Path('model_reports/plots')
plot_dir.mkdir(parents=True, exist_ok=True)

# Plot Gini/Gains curve
for year, result in zip([develop_year, validate_year], results):
    gains_path = plot_gains(
        y_true=df[df['score_year'] == year]['default_flag'],
        y_score=df[df['score_year'] == year]['score_pd'],
        title=f'Gains Curve {year}',
        out_path=plot_dir / f'mr_gains_curve_{year}.png'
    )
    display(Image(filename=str(gains_path)))

# Plot AUC ROC
for year, result in zip([develop_year, validate_year], results):
    roc_path = plot_roc(
        y_true=df[df['score_year'] == year]['default_flag'],
        y_score=df[df['score_year'] == year]['score_pd'],
        title=f'AUC ROC {year}',
        out_path=plot_dir / f'mr_auc_roc_{year}.png'
    )
    display(Image(filename=str(roc_path)))

# Plot KS statistic
for year, result in zip([develop_year, validate_year], results):
    ks_path, ks_table = plot_ks_cdf_with_maxgap(
        y_true=df[df['score_year'] == year]['default_flag'],
        y_score=df[df['score_year'] == year]['score_pd'],
        title=f'KS CDF {year}',
        out_path=plot_dir / f'mr_ks_cdf_{year}.png'
    )
    display(Image(filename=str(ks_path)))